In [1]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# plotting 설정
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

try:
    # 일반 Python 스크립트 실행 시 (__file__이 존재)
    current_path = Path(__file__).resolve()
except NameError:
    # Jupyter Notebook 실행 시 (__file__ 없음)
    current_path = Path().resolve()

# 현재 경로에서 stock_forecast 폴더까지 자동 탐색
for parent in current_path.parents:
    if (parent / "stock_forecast" / "DATA").is_dir():
        stock_forecast_path = parent / "stock_forecast"
        break
else:
    raise ImportError("stock_forecast/DATA 폴더를 찾을 수 없습니다.")

# sys.path에 추가
if str(stock_forecast_path) not in sys.path:
    sys.path.insert(0, str(stock_forecast_path))

print(f"sys.path에 등록된 경로: {stock_forecast_path}")


from datetime import datetime, timedelta
from typing import Iterable

from sklearn.preprocessing import StandardScaler, RobustScaler
from tqdm import tqdm

# 사용자 유틸 함수들
from DATA.stock_invest_function import *
from datetime import datetime, timedelta

# SQLAlchemy
from sqlalchemy import create_engine, text, Table, MetaData
from sqlalchemy.dialects.mysql import insert as mysql_insert

# -----------------------------
# Utility Functions for Data Cleaning
# -----------------------------
def clean_numeric_data(series, method='drop'):
    """
    숫자 데이터에서 inf, -inf, NaN 값을 처리

    Parameters:
    - series: pandas Series
    - method: 'drop', 'fill_median', 'fill_mean', 'fill_zero'
    """
    # inf, -inf를 NaN으로 변환
    series = series.replace([np.inf, -np.inf], np.nan)

    if method == 'drop':
        return series.dropna()
    elif method == 'fill_median':
        return series.fillna(series.median())
    elif method == 'fill_mean':
        return series.fillna(series.mean())
    elif method == 'fill_zero':
        return series.fillna(0)
    else:
        return series

def validate_dataframe(df, name="DataFrame"):
    """데이터프레임의 inf/NaN 값 상태를 확인하고 리포트"""
    print(f"\n=== {name} 데이터 품질 체크 ===")

    for col in df.select_dtypes(include=[np.number]).columns:
        inf_count = np.isinf(df[col]).sum()
        nan_count = df[col].isna().sum()
        total_count = len(df)

        if inf_count > 0 or nan_count > 0:
            print(f"{col}: inf={inf_count}, NaN={nan_count} (전체 {total_count}건 중)")

        # inf 값이 있으면 경고
        if inf_count > 0:
            print(f"⚠️  {col}에 inf 값이 {inf_count}개 있습니다!")

    return df

def safe_percentage_change(series, periods=1):
    """안전한 퍼센트 변화 계산 (inf/NaN 처리 포함)"""
    pct_change = series.pct_change(periods=periods)

    # inf, -inf를 NaN으로 변환
    pct_change = pct_change.replace([np.inf, -np.inf], np.nan)

    # 극단적인 값들 처리 (1000% 이상 변화는 이상치로 간주)
    pct_change = pct_change.clip(lower=-10, upper=10)

    return pct_change

def safe_standardization(data, method='robust'):
    """안전한 표준화 (inf/NaN 처리 포함)"""
    # inf, NaN 값 제거
    clean_data = clean_numeric_data(data, method='fill_median')

    if method == 'robust':
        scaler = RobustScaler()  # 이상치에 더 강건
    else:
        scaler = StandardScaler()

    # 2차원 배열로 변환
    data_2d = clean_data.values.reshape(-1, 1)

    # 표준화 수행
    scaled_data = scaler.fit_transform(data_2d).flatten()

    # 결과도 한번 더 체크
    scaled_series = pd.Series(scaled_data, index=clean_data.index)
    scaled_series = clean_numeric_data(scaled_series, method='fill_zero')

    return scaled_series

# -----------------------------
# Parameters (사용자 입력 부분)
# -----------------------------
# 사용자가 수정할 부분
tic_name = 'A000660'              # 한국 기업 코드 (예: 삼성전자)
hs_code = '854231'               # 외생변수 HS CODE (수출 데이터용)
st_date = '2010-01-01'
end_date = '2025-07-31'
today_date = pd.to_datetime(datetime.today().date())

USE_EXOGENOUS = True             # 외생변수 사용 여부

# 데이터베이스 정보
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

# -----------------------------
# Enhanced SARIMA Forecasting Functions
# -----------------------------
def enhanced_sarima_forecast_korea(data, exog_col=None, forecast_steps=4, use_log=False):
    """Enhanced SARIMA forecasting for Korean companies"""
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from itertools import product
    import warnings
    warnings.filterwarnings('ignore')

    df = data.copy().sort_values('date')
    endog = df['endog_var']

    # endog 데이터 정리
    endog = clean_numeric_data(endog, method='fill_median')

    if len(endog) < 8:  # 최소 데이터 요구사항
        raise ValueError(f"데이터가 부족합니다: {len(endog)}건 (최소 8건 필요)")

    exog = None
    if exog_col and exog_col in df.columns:
        exog = df[exog_col]
        exog = clean_numeric_data(exog, method='fill_median')

        # endog와 길이 맞추기
        min_len = min(len(endog), len(exog))
        endog = endog.iloc[-min_len:]
        exog = exog.iloc[-min_len:]

    if use_log:
        endog = np.log(endog.clip(lower=0.01))  # 로그 변환 시 0 방지
        endog = clean_numeric_data(endog, method='fill_median')

    # SARIMA 파라미터 범위 (한국 분기 데이터에 최적화)
    p_values = [0, 1, 2]
    d_values = [0, 1]
    q_values = [0, 1, 2]
    P_values = [0, 1]
    D_values = [0, 1]
    Q_values = [0, 1]
    s_value = 4  # 분기 계절성

    best_aic = float('inf')
    best_params = None
    best_model = None

    for p, d, q, P, D, Q in product(p_values, d_values, q_values, P_values, D_values, Q_values):
        try:
            total_params = p + q + P + Q + 1
            if total_params >= len(endog) * 0.3:
                continue

            model = SARIMAX(
                endog, exog=exog, order=(p, d, q),
                seasonal_order=(P, D, Q, s_value),
                enforce_stationarity=False, enforce_invertibility=False
            )
            fitted_model = model.fit(disp=False, maxiter=100)

            if np.isfinite(fitted_model.aic) and fitted_model.aic < best_aic:
                best_aic = fitted_model.aic
                best_params = (p, d, q, P, D, Q, s_value)
                best_model = fitted_model
        except Exception:
            continue

    if best_model is None:
        try:
            model = SARIMAX(endog, exog=exog, order=(1, 1, 1),
                          seasonal_order=(0, 0, 0, 0),
                          enforce_stationarity=False, enforce_invertibility=False)
            best_model = model.fit(disp=False)
            best_params = (1, 1, 1, 0, 0, 0, 0)
        except:
            raise ValueError("All SARIMA configurations failed")

    # 예측 수행
    if exog is not None:
        last_exog_value = exog.iloc[-1]
        if np.isfinite(last_exog_value):
            future_exog = [last_exog_value] * forecast_steps
        else:
            future_exog = [0] * forecast_steps
        forecast = best_model.forecast(steps=forecast_steps, exog=future_exog)
    else:
        forecast = best_model.forecast(steps=forecast_steps)

    # 예측 결과 정리
    forecast = clean_numeric_data(pd.Series(forecast), method='fill_median')

    if use_log:
        forecast = np.exp(forecast)

    param_string = f"({best_params[0]},{best_params[1]},{best_params[2]})({best_params[3]},{best_params[4]},{best_params[5]},{best_params[6]})"
    return forecast, param_string

def forecast_monthly_sarima_korea(data, exog_col=None, forecast_steps=12):
    """Monthly SARIMA forecasting for Korean PSR"""
    try:
        forecast, params = enhanced_sarima_forecast_korea(data, exog_col, forecast_steps, use_log=False)
        last_date = pd.to_datetime(data['date'].iloc[-1])
        forecast_dates = pd.date_range(start=last_date + pd.DateOffset(months=1),
                                     periods=forecast_steps, freq='M')
        forecast_df = pd.DataFrame({'date': forecast_dates, 'forecast': forecast.values})
        return forecast_df, params
    except Exception as e:
        print(f"Monthly SARIMA failed: {e}")
        return None, None

# -----------------------------
# Database Helper Functions
# -----------------------------
def _table_exists(conn, db_name: str, tname: str) -> bool:
    return bool(conn.execute(text("""
        SELECT COUNT(*) FROM information_schema.tables
        WHERE table_schema = :s AND table_name = :t
    """), {"s": db_name, "t": tname}).scalar())

def _column_exists(conn, db_name: str, tname: str, col: str) -> bool:
    return bool(conn.execute(text("""
        SELECT COUNT(*) FROM information_schema.columns
        WHERE table_schema = :s AND table_name = :t AND column_name = :c
    """), {"s": db_name, "t": tname, "c": col}).scalar())

def _index_exists(conn, db_name: str, tname: str, idx: str) -> bool:
    return bool(conn.execute(text("""
        SELECT COUNT(*) FROM information_schema.statistics
        WHERE table_schema = :s AND table_name = :t AND index_name = :i
    """), {"s": db_name, "t": tname, "i": idx}).scalar())

def _ensure_korea_table_and_indexes(engine, table_name: str, db_name: str):
    """한국 기업 밸류에이션 테이블 스키마 보장"""
    with engine.begin() as conn:
        # 1) 테이블 생성
        if not _table_exists(conn, db_name, table_name):
            conn.execute(text(f"""
                CREATE TABLE `{table_name}` (
                  id BIGINT AUTO_INCREMENT PRIMARY KEY,
                  ticker         VARCHAR(32)  NOT NULL,
                  forecast_date  DATE         NOT NULL,
                  target_date    DATE         NOT NULL,
                  indicator      VARCHAR(128) NOT NULL,
                  frequency      VARCHAR(8)   NOT NULL,
                  value          DOUBLE       NULL,
                  exog_var       VARCHAR(64)  NULL,
                  params         VARCHAR(255) NULL,
                  valuation_time DATETIME     NOT NULL,
                  created_at     TIMESTAMP    DEFAULT CURRENT_TIMESTAMP,
                  INDEX `ix_ticker`        (ticker),
                  INDEX `ix_forecast_date` (forecast_date),
                  INDEX `ix_target_date`   (target_date),
                  INDEX `ix_indicator`     (indicator),
                  INDEX `ix_frequency`     (frequency)
                ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
            """))
            print(f"테이블 {table_name} 생성 완료")

        # 2) 기존 잘못된 데이터 정리
        print("기존 잘못된 데이터 정리 중...")
        try:
            cleanup_query = f"""
                DELETE FROM `{table_name}`
                WHERE target_date IS NULL
                   OR target_date = '0000-00-00'
                   OR target_date < '1900-01-01'
            """
            result = conn.execute(text(cleanup_query))
            if result.rowcount > 0:
                print(f"잘못된 데이터 {result.rowcount}건 삭제됨")
        except Exception as e:
            print(f"데이터 정리 중 오류: {e}")

        # 3) 누락 컬럼 보강
        for col, ddl in [
            ("forecast_date", "ADD COLUMN forecast_date DATE NOT NULL"),
            ("target_date",   "ADD COLUMN target_date DATE NOT NULL"),
            ("indicator",     "ADD COLUMN indicator VARCHAR(128) NOT NULL"),
            ("frequency",     "ADD COLUMN frequency VARCHAR(8) NOT NULL"),
            ("value",         "ADD COLUMN value DOUBLE NULL"),
            ("exog_var",      "ADD COLUMN exog_var VARCHAR(64) NULL"),
            ("params",        "ADD COLUMN params VARCHAR(255) NULL"),
            ("valuation_time","ADD COLUMN valuation_time DATETIME NOT NULL"),
        ]:
            if not _column_exists(conn, db_name, table_name, col):
                try:
                    conn.execute(text(f"ALTER TABLE `{table_name}` {ddl}"))
                except Exception as e:
                    print(f"컬럼 {col} 추가 중 오류: {e}")

        # 4) UNIQUE 인덱스 생성
        uniq = f"ux_{table_name}_keytime"
        try:
            if _index_exists(conn, db_name, table_name, uniq):
                conn.execute(text(f"DROP INDEX `{uniq}` ON `{table_name}`"))
                print(f"기존 인덱스 {uniq} 삭제됨")

            conn.execute(text(f"""
                CREATE UNIQUE INDEX `{uniq}`
                ON `{table_name}` (ticker, forecast_date, indicator, frequency, target_date, valuation_time)
            """))
            print(f"새 인덱스 {uniq} 생성됨")

        except Exception as e:
            print(f"인덱스 생성 중 오류: {e}")

# -----------------------------
# Data Preparation
# -----------------------------
print("=" * 50)
print(f"한국 기업 밸류에이션 시스템 시작")
print(f"대상 기업: {tic_name}")
print(f"외생변수 HS코드: {hs_code}")
print(f"외생변수 사용: {'예' if USE_EXOGENOUS else '아니오'}")
print("=" * 50)

print("1. 기본 데이터 로딩...")
# 한국 재무 데이터 로딩
fs_df = fetch_table_data(db_info, "korea_fs_data")
fs_df.rename(columns={'Date': 'date'}, inplace=True)

# 매출액 데이터 필터링
target_indicator = '매출액(천원)'
filtered_df = fs_df[fs_df['indicator'] == target_indicator].copy()

# 날짜 정제 및 정렬
filtered_df['date'] = pd.to_datetime(filtered_df['date'])
filtered_df.sort_values(by='date', inplace=True)

# value 컬럼 타입 변환 및 정리
if 'value' not in filtered_df.columns:
    raise KeyError("'value' 컬럼이 없습니다.")

filtered_df['value'] = pd.to_numeric(filtered_df['value'], errors='coerce')
filtered_df['value'] = clean_numeric_data(filtered_df['value'], method='drop')

# 피벗 테이블 생성 (행: date, 열: Symbol, 값: value)
pivot_df = filtered_df.pivot_table(
    index='date',
    columns='symbol',
    values='value',
    aggfunc='first'
)

# 대상 기업 매출 데이터 추출
if tic_name not in pivot_df.columns:
    raise KeyError(f"기업 코드 '{tic_name}'의 데이터가 없습니다.")

endog_df = pivot_df[[tic_name]].reset_index()
endog_df = endog_df.dropna()
endog_df = endog_df[(endog_df['date'] >= st_date) & (endog_df['date'] <= end_date)]

# 매출 데이터 검증
endog_df[tic_name] = clean_numeric_data(endog_df[tic_name], method='fill_median')
validate_dataframe(endog_df, "매출 데이터")

print(f"매출 데이터 로딩 완료: {len(endog_df)}건 ({endog_df['date'].min():%Y-%m} ~ {endog_df['date'].max():%Y-%m})")

# 2. 외생변수 데이터 준비
quarterly_sum_df = None
if USE_EXOGENOUS:
    print("2. 외생변수 데이터 준비...")
    try:
        # 수출 예측 데이터 로딩
        temp_df = load_forecast_by_hscode(db_info, hs_code, table_name='korea_monthly_trade_data_forecast')

        if temp_df is not None and len(temp_df) > 0:
            print(f"외생변수 원본 데이터: {len(temp_df)}건")

            # 데이터 품질 체크
            validate_dataframe(temp_df, "외생변수 원본")

            # 중복 제거
            target_export_df = temp_df.drop_duplicates(subset=['date'])

            # date를 datetime으로 변환
            target_export_df['date'] = pd.to_datetime(target_export_df['date'])

            # 외생변수 값 정리
            target_export_df['expDlr_forecast_12m'] = clean_numeric_data(
                target_export_df['expDlr_forecast_12m'], method='fill_median'
            )

            # 분기 추출 및 그룹화
            target_export_df['quarter'] = target_export_df['date'].dt.to_period('Q')
            quarterly_sum_df = target_export_df.groupby('quarter')['expDlr_forecast_12m'].sum().reset_index()

            # datetime 변환 후 추가 계산
            quarterly_sum_df['quarter'] = quarterly_sum_df['quarter'].dt.to_timestamp()

            # 안전한 퍼센트 변화 계산
            quarterly_sum_df['export_qoq_change'] = safe_percentage_change(
                quarterly_sum_df['expDlr_forecast_12m'], periods=1
            )
            quarterly_sum_df['export_yoy_change'] = safe_percentage_change(
                quarterly_sum_df['expDlr_forecast_12m'], periods=4
            )

            quarterly_sum_df['date_month'] = quarterly_sum_df['quarter'] + pd.offsets.QuarterEnd(0)
            quarterly_sum_df['date'] = pd.to_datetime(quarterly_sum_df['date_month'])
            quarterly_sum_df.set_index('date', inplace=True)
            quarterly_sum_df.drop(columns='quarter', inplace=True)
            quarterly_sum_df = quarterly_sum_df.reset_index()

            # 외생변수 검증
            validate_dataframe(quarterly_sum_df, "분기별 집계 외생변수")

            # 안전한 스케일링
            quarterly_sum_df['exog_scaled'] = safe_standardization(
                quarterly_sum_df['export_yoy_change'], method='robust'
            )

            # 최종 검증
            validate_dataframe(quarterly_sum_df, "스케일링된 외생변수")

            print(f"외생변수 데이터 준비 완료: {len(quarterly_sum_df)}건")
        else:
            print("외생변수 데이터가 없어 사용하지 않습니다.")
            USE_EXOGENOUS = False
    except Exception as e:
        print(f"외생변수 데이터 로딩 실패: {e}")
        import traceback
        traceback.print_exc()
        USE_EXOGENOUS = False

# 3. PSR 데이터 준비 (한국 기업용)
print("3. PSR 데이터 준비...")

# 변수 초기화 (PyCharm 경고 방지)
endog_ratio_df = None
merged_ratio_exog = None

# 한국 기업의 경우 시가총액과 매출액으로 PSR 계산
try:
    # 시가총액 데이터 가져오기 (예시 - 실제 구현에 맞게 수정 필요)
    market_cap_df = fetch_table_data(db_info, "ks_listed_company_daily_marketcap")  # 테이블명은 실제에 맞게 수정
    market_cap_df = market_cap_df[market_cap_df['symbol'] == tic_name].copy()
    market_cap_df['date'] = pd.to_datetime(market_cap_df['date'])

    # 시가총액 데이터 정리
    market_cap_df['market_cap'] = clean_numeric_data(market_cap_df['market_cap'], method='fill_median')

    # PSR 계산을 위한 데이터 병합
    merged_for_psr = pd.merge_asof(
        market_cap_df.sort_values('date'),
        endog_df.sort_values('date'),
        on='date',
        direction='backward'
    )

    # PSR 계산 (분모가 0이 되지 않도록 보호)
    merged_for_psr['PSR'] = merged_for_psr['market_cap'] / (merged_for_psr[tic_name] + 1e-10)
    merged_for_psr['PSR'] = clean_numeric_data(merged_for_psr['PSR'], method='fill_median')
    merged_for_psr = merged_for_psr.dropna(subset=['PSR'])

    endog_ratio_df = merged_for_psr[['date', 'PSR']].copy()
    validate_dataframe(endog_ratio_df, "PSR 데이터")
    print(f"PSR 데이터 계산 완료: {len(endog_ratio_df)}건")

except Exception as e:
    print(f"PSR 데이터 계산 실패: {e}")
    # 대체 방법: 고정 PSR 값 사용
    endog_ratio_df = endog_df[['date']].copy()
    endog_ratio_df['PSR'] = 2.0  # 기본 PSR 값
    print("기본 PSR 값 2.0 사용")

# -----------------------------
# Build Datasets
# -----------------------------
print("4. 예측 데이터셋 구성...")

# 변수 초기화 (PyCharm 경고 방지)
merged_revenue = None

# Revenue forecasting dataset
if USE_EXOGENOUS and quarterly_sum_df is not None:
    merged_revenue = merge_endog_exog_data(
        endog_df=endog_df,
        exog_df=quarterly_sum_df,
        endog_col=tic_name,
        exog_col='export_yoy_change',
        start_date=st_date,
        end_date=end_date
    )
    merged_revenue = merged_revenue.drop_duplicates(subset=['date'], keep='first').reset_index(drop=True)

    # 최종 데이터 정리
    merged_revenue['endog_var'] = clean_numeric_data(merged_revenue['endog_var'], method='fill_median')
    merged_revenue['exog_var'] = clean_numeric_data(merged_revenue['exog_var'], method='fill_median')

    validate_dataframe(merged_revenue, "매출 예측 데이터셋 (외생변수 포함)")
    print(f"매출 예측 데이터셋 (외생변수 포함): {len(merged_revenue)}건")
else:
    merged_revenue = endog_df.copy()
    merged_revenue.rename(columns={tic_name: 'endog_var'}, inplace=True)
    merged_revenue['exog_var'] = None
    merged_revenue['endog_var'] = clean_numeric_data(merged_revenue['endog_var'], method='fill_median')
    print(f"매출 예측 데이터셋 (외생변수 미포함): {len(merged_revenue)}건")

# PSR forecasting dataset 초기화
merged_ratio_exog = None

# PSR forecasting dataset 구성
if USE_EXOGENOUS and quarterly_sum_df is not None and endog_ratio_df is not None:
    merged_ratio_exog = pd.merge(
        endog_ratio_df,
        quarterly_sum_df[['date', 'exog_scaled']],
        on='date', how='outer'
    ).sort_values('date')

    merged_ratio_exog['PSR'] = merged_ratio_exog['PSR'].interpolate(method='linear')
    merged_ratio_exog['exog_scaled'] = (merged_ratio_exog['exog_scaled']
                                       .fillna(method='ffill').fillna(method='bfill'))
    merged_ratio_exog = merged_ratio_exog.dropna()

    # 데이터 정리
    merged_ratio_exog['PSR'] = clean_numeric_data(merged_ratio_exog['PSR'], method='fill_median')
    merged_ratio_exog['exog_scaled'] = clean_numeric_data(merged_ratio_exog['exog_scaled'], method='fill_median')

    merged_ratio_exog.rename(columns={'PSR':'endog_var','exog_scaled':'exog_var'}, inplace=True)
    validate_dataframe(merged_ratio_exog, "PSR 예측 데이터셋 (외생변수 포함)")
    print(f"PSR 예측 데이터셋 (외생변수 포함): {len(merged_ratio_exog)}건")
elif endog_ratio_df is not None:
    merged_ratio_exog = endog_ratio_df.copy()
    merged_ratio_exog.rename(columns={'PSR':'endog_var'}, inplace=True)
    merged_ratio_exog['exog_var'] = None
    merged_ratio_exog['endog_var'] = clean_numeric_data(merged_ratio_exog['endog_var'], method='fill_median')
    print(f"PSR 예측 데이터셋 (외생변수 미포함): {len(merged_ratio_exog)}건")
else:
    # endog_ratio_df가 None인 경우 기본 데이터 생성
    print("PSR 데이터가 없어 기본 PSR 데이터셋을 생성합니다.")
    merged_ratio_exog = endog_df[['date']].copy()
    merged_ratio_exog['endog_var'] = 2.0  # 기본 PSR 값
    merged_ratio_exog['exog_var'] = None
    print(f"기본 PSR 예측 데이터셋: {len(merged_ratio_exog)}건")

# 데이터 무결성 검증
if merged_revenue is None:
    raise ValueError("매출 예측 데이터셋이 생성되지 않았습니다.")
if merged_ratio_exog is None:
    raise ValueError("PSR 예측 데이터셋이 생성되지 않았습니다.")

# -----------------------------
# Forecasting
# -----------------------------
print("5. 예측 수행...")

# Revenue forecasting
print("5-1. 매출 예측 (SARIMA)...")
revenue_forecasts_with_exog = None
revenue_params_with_exog = None
revenue_forecasts_without_exog = None
revenue_params_without_exog = None

if USE_EXOGENOUS and 'exog_var' in merged_revenue.columns and merged_revenue['exog_var'].notna().any():
    try:
        revenue_forecasts_with_exog, revenue_params_with_exog = enhanced_sarima_forecast_korea(
            merged_revenue, exog_col='exog_var', forecast_steps=4, use_log=True
        )
        print(f"매출 예측 (외생변수 포함) - 파라미터: {revenue_params_with_exog}")
    except Exception as e:
        print(f"매출 외생변수 예측 실패: {e}")

try:
    revenue_forecasts_without_exog, revenue_params_without_exog = enhanced_sarima_forecast_korea(
        merged_revenue, exog_col=None, forecast_steps=4, use_log=True
    )
    print(f"매출 예측 (외생변수 미포함) - 파라미터: {revenue_params_without_exog}")
except Exception as e:
    print(f"매출 예측 실패: {e}")
    last_revenue = merged_revenue['endog_var'].iloc[-1]
    revenue_forecasts_without_exog = pd.Series([last_revenue * 1.05] * 4)
    revenue_params_without_exog = "fallback"

# PSR forecasting (분기별)
print("5-2. PSR 예측 - 분기별 (SARIMA)...")
ratio_forecasts_with_exog = None
ratio_params_with_exog = None
ratio_forecasts_without_exog = None
ratio_params_without_exog = None

if USE_EXOGENOUS and 'exog_var' in merged_ratio_exog.columns and merged_ratio_exog['exog_var'].notna().any():
    try:
        ratio_forecasts_with_exog, ratio_params_with_exog = enhanced_sarima_forecast_korea(
            merged_ratio_exog, exog_col='exog_var', forecast_steps=4, use_log=False
        )
        print(f"PSR 분기별 예측 (외생변수 포함) - 파라미터: {ratio_params_with_exog}")
    except Exception as e:
        print(f"PSR 외생변수 예측 실패: {e}")

try:
    ratio_forecasts_without_exog, ratio_params_without_exog = enhanced_sarima_forecast_korea(
        merged_ratio_exog, exog_col=None, forecast_steps=4, use_log=False
    )
    print(f"PSR 분기별 예측 (외생변수 미포함) - 파라미터: {ratio_params_without_exog}")
except Exception as e:
    print(f"PSR 분기별 예측 실패: {e}")
    last_psr = merged_ratio_exog['endog_var'].iloc[-1]
    ratio_forecasts_without_exog = pd.Series([last_psr] * 4)
    ratio_params_without_exog = "fallback"

# PSR forecasting (월별)
print("5-3. PSR 예측 - 월별 (SARIMA)...")
# 월별 데이터 준비
monthly_psr_data = merged_ratio_exog.copy()
monthly_psr_data['date'] = pd.to_datetime(monthly_psr_data['date'])

# 월별 예측을 위한 보간
monthly_psr_interpolated = monthly_psr_data.set_index('date').resample('M').interpolate().reset_index()

# 보간된 데이터 정리
monthly_psr_interpolated['endog_var'] = clean_numeric_data(monthly_psr_interpolated['endog_var'], method='fill_median')
if 'exog_var' in monthly_psr_interpolated.columns:
    monthly_psr_interpolated['exog_var'] = clean_numeric_data(monthly_psr_interpolated['exog_var'], method='fill_median')

validate_dataframe(monthly_psr_interpolated, "월별 보간된 PSR 데이터")

ratio_monthly_forecasts_with_exog = None
ratio_monthly_params_with_exog = None
ratio_monthly_forecasts_without_exog = None
ratio_monthly_params_without_exog = None

if USE_EXOGENOUS and 'exog_var' in monthly_psr_interpolated.columns and monthly_psr_interpolated['exog_var'].notna().any():
    try:
        ratio_monthly_forecasts_with_exog, ratio_monthly_params_with_exog = forecast_monthly_sarima_korea(
            monthly_psr_interpolated, exog_col='exog_var', forecast_steps=12
        )
        print(f"PSR 월별 예측 (외생변수 포함) - 파라미터: {ratio_monthly_params_with_exog}")
    except Exception as e:
        print(f"PSR 월별 외생변수 예측 실패: {e}")

try:
    ratio_monthly_forecasts_without_exog, ratio_monthly_params_without_exog = forecast_monthly_sarima_korea(
        monthly_psr_interpolated, exog_col=None, forecast_steps=12
    )
    print(f"PSR 월별 예측 (외생변수 미포함) - 파라미터: {ratio_monthly_params_without_exog}")
except Exception as e:
    print(f"PSR 월별 예측 실패: {e}")
    last_date = pd.to_datetime(monthly_psr_interpolated['date'].iloc[-1])
    last_value = monthly_psr_interpolated['endog_var'].iloc[-1]
    forecast_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=12, freq='M')
    ratio_monthly_forecasts_without_exog = pd.DataFrame({'date': forecast_dates, 'forecast': [last_value] * 12})
    ratio_monthly_params_without_exog = "fallback"

# -----------------------------
# Valuation Assembly
# -----------------------------
print("6. 밸류에이션 결과 조합...")

# Forward revenue calculation
if revenue_forecasts_with_exog is not None:
    forward_revenue_with_exog = revenue_forecasts_with_exog.sum()
    revenue_forecasts_with_exog = revenue_forecasts_with_exog.rename('revenue_with_exog')
else:
    forward_revenue_with_exog = None

forward_revenue_without_exog = revenue_forecasts_without_exog.sum()
revenue_forecasts_without_exog = revenue_forecasts_without_exog.rename('revenue_without_exog')

print(f"향후 4분기 매출 예측:")
if forward_revenue_with_exog is not None:
    print(f"  외생변수 포함: {forward_revenue_with_exog/1e6:.1f}억원")
print(f"  외생변수 미포함: {forward_revenue_without_exog/1e6:.1f}억원")

# Long-term (Quarterly) valuation
print("분기별 장기 밸류에이션 구성...")
longterm_series_data, longterm_names = [], []

if ratio_forecasts_with_exog is not None:
    longterm_series_data.extend([ratio_forecasts_with_exog, revenue_forecasts_with_exog])
    longterm_names.extend(['PSR_quarter_with_exog', 'revenue_with_exog'])

longterm_series_data.extend([ratio_forecasts_without_exog, revenue_forecasts_without_exog])
longterm_names.extend(['PSR_quarter_without_exog', 'revenue_without_exog'])

longterm_value_forecasts = pd.concat(longterm_series_data, axis=1, keys=longterm_names)

if forward_revenue_with_exog is not None:
    longterm_value_forecasts['forward_revenue_with_exog'] = forward_revenue_with_exog
longterm_value_forecasts['forward_revenue_without_exog'] = forward_revenue_without_exog

# Enterprise value calculation
if ratio_forecasts_with_exog is not None:
    longterm_value_forecasts['value_with_exog'] = (
        longterm_value_forecasts['PSR_quarter_with_exog'] * longterm_value_forecasts['forward_revenue_with_exog']
    )
longterm_value_forecasts['value_without_exog'] = (
    longterm_value_forecasts['PSR_quarter_without_exog'] * longterm_value_forecasts['forward_revenue_without_exog']
)

# 밸류에이션 결과 정리
for col in longterm_value_forecasts.select_dtypes(include=[np.number]).columns:
    longterm_value_forecasts[col] = clean_numeric_data(longterm_value_forecasts[col], method='fill_median')

# target_date 설정 (분기별)
last_q_obs = max(pd.to_datetime(endog_ratio_df['date']).max(),
                 pd.to_datetime(endog_df['date']).max())
q_dates = pd.date_range(start=last_q_obs + pd.offsets.QuarterEnd(1),
                        periods=len(longterm_value_forecasts), freq='Q')
longterm_value_forecasts = longterm_value_forecasts.reset_index(drop=True)
longterm_value_forecasts['target_date'] = q_dates
longterm_value_forecasts['frequency'] = 'Q'
longterm_value_forecasts['ticker'] = tic_name
longterm_value_forecasts['forecast_date'] = today_date

# Mid-term (Monthly) valuation
print("월별 중기 밸류에이션 구성...")
midterm_series_data, midterm_names = [], []

if ratio_monthly_forecasts_with_exog is not None:
    # forecast 데이터 정리
    forecast_data = clean_numeric_data(ratio_monthly_forecasts_with_exog['forecast'], method='fill_median')
    midterm_series_data.append(forecast_data)
    midterm_names.append('PSR_monthly_sarima_with_exog')

forecast_data_without = clean_numeric_data(ratio_monthly_forecasts_without_exog['forecast'], method='fill_median')
midterm_series_data.append(forecast_data_without)
midterm_names.append('PSR_monthly_sarima_without_exog')

midterm_value_forecasts = pd.concat(midterm_series_data, axis=1, keys=midterm_names)

if ratio_monthly_forecasts_with_exog is not None:
    midterm_value_forecasts['forward_revenue_with_exog'] = forward_revenue_with_exog
midterm_value_forecasts['forward_revenue_without_exog'] = forward_revenue_without_exog

# target_date 설정 (월별)
if ratio_monthly_forecasts_with_exog is not None:
    src_dates = ratio_monthly_forecasts_with_exog['date'].values
else:
    src_dates = ratio_monthly_forecasts_without_exog['date'].values

midterm_value_forecasts = midterm_value_forecasts.reset_index(drop=True)
midterm_value_forecasts['target_date'] = src_dates
midterm_value_forecasts['frequency'] = 'M'
midterm_value_forecasts['ticker'] = tic_name
midterm_value_forecasts['forecast_date'] = today_date

# Enterprise value calculation (월별)
if ratio_monthly_forecasts_with_exog is not None:
    midterm_value_forecasts['value_monthly_sarima_with_exog'] = (
        midterm_value_forecasts['PSR_monthly_sarima_with_exog'] *
        midterm_value_forecasts['forward_revenue_with_exog']
    )
midterm_value_forecasts['value_monthly_sarima_without_exog'] = (
    midterm_value_forecasts['PSR_monthly_sarima_without_exog'] *
    midterm_value_forecasts['forward_revenue_without_exog']
)

# 월별 밸류에이션 결과 정리
for col in midterm_value_forecasts.select_dtypes(include=[np.number]).columns:
    midterm_value_forecasts[col] = clean_numeric_data(midterm_value_forecasts[col], method='fill_median')

# 최종 검증
validate_dataframe(longterm_value_forecasts, "분기별 밸류에이션 결과")
validate_dataframe(midterm_value_forecasts, "월별 밸류에이션 결과")

# -----------------------------
# Melt & Combine Results
# -----------------------------
print("7. 결과 변환 및 결합...")

def melt_forecast_df_korea(df, freq, params_dict=None):
    """한국 기업용 forecast 데이터 melt 함수"""
    melted = df.melt(
        id_vars=['frequency','ticker','forecast_date','target_date'],
        var_name='indicator',
        value_name='value'
    )
    melted['exog_var'] = hs_code if USE_EXOGENOUS else None
    melted['params'] = None

    if params_dict:
        for indicator, param in params_dict.items():
            if param:
                melted.loc[melted['indicator'] == indicator, 'params'] = param

    # 최종 value 정리
    melted['value'] = clean_numeric_data(melted['value'], method='fill_median')

    return melted

# 파라미터 사전 구성
longterm_params = {}
if revenue_params_with_exog:
    longterm_params['revenue_with_exog'] = revenue_params_with_exog
if revenue_params_without_exog:
    longterm_params['revenue_without_exog'] = revenue_params_without_exog
if ratio_params_with_exog:
    longterm_params['PSR_quarter_with_exog'] = ratio_params_with_exog
if ratio_params_without_exog:
    longterm_params['PSR_quarter_without_exog'] = ratio_params_without_exog

midterm_params = {}
if ratio_monthly_params_with_exog:
    midterm_params['PSR_monthly_sarima_with_exog'] = ratio_monthly_params_with_exog
if ratio_monthly_params_without_exog:
    midterm_params['PSR_monthly_sarima_without_exog'] = ratio_monthly_params_without_exog

# Melt 수행
longterm_melted = melt_forecast_df_korea(longterm_value_forecasts, 'Q', longterm_params)
midterm_melted = melt_forecast_df_korea(midterm_value_forecasts, 'M', midterm_params)

# 최종 결합
combined_long_format = pd.concat([longterm_melted, midterm_melted],
                                axis=0, ignore_index=True)

# 최종 검증
validate_dataframe(combined_long_format, "최종 결합 결과")

print("한국 기업 밸류에이션 예측 완료!")
print(f"결과 요약:")
print(f"  총 레코드 수: {len(combined_long_format)}")
print(f"  외생변수 사용: {'예 (' + hs_code + ')' if USE_EXOGENOUS else '아니오'}")
print(f"  예측 주기: {combined_long_format['frequency'].unique()}")
print(f"  고유 지표 수: {len(combined_long_format['indicator'].unique())}")

# 샘플 기업가치 출력
print("\n샘플 기업가치 예측:")
value_indicators = combined_long_format[combined_long_format['indicator'].str.contains('value', na=False)]
if len(value_indicators) > 0:
    sample_valuations = value_indicators.head(8)[['indicator','value','params','frequency','target_date']]
    for _, row in sample_valuations.iterrows():
        value_billions = row['value'] / 1e8 if pd.notna(row['value']) else 0  # 억원 단위
        target_date_str = pd.to_datetime(row['target_date']).strftime('%Y-%m')
        print(f"  {row['indicator']}: {value_billions:.1f}억원 ({row['params']}, {row['frequency']}, {target_date_str})")

print(f"\n결과 데이터 구조:")
print("컬럼:", list(combined_long_format.columns))
print("\n지표 유형:")
unique_indicators = combined_long_format['indicator'].unique()
for indicator in sorted(unique_indicators):
    count = len(combined_long_format[combined_long_format['indicator'] == indicator])
    print(f"  {indicator}: {count}개 레코드")

# -----------------------------
# Database Save Function
# -----------------------------
def save_korea_valuation_to_db(db_info, combined_long_format):
    """한국 기업 밸류에이션 결과를 데이터베이스에 저장"""
    try:
        from sqlalchemy import create_engine, text
        from datetime import datetime, timedelta
        import pandas as pd

        connection_string = (
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )

        engine = create_engine(connection_string)
        table_name = 'korea_company_valuation_result'

        print("=== 데이터베이스 저장 시작 ===")
        df_to_save = combined_long_format.copy()

        # 최종 데이터 정리
        for col in df_to_save.select_dtypes(include=[np.number]).columns:
            df_to_save[col] = clean_numeric_data(df_to_save[col], method='fill_median')

        # 날짜 컬럼 정리
        if 'target_date' in df_to_save.columns:
            df_to_save['target_date'] = pd.to_datetime(df_to_save['target_date'])
        if 'forecast_date' in df_to_save.columns:
            df_to_save['forecast_date'] = pd.to_datetime(df_to_save['forecast_date'])

        # NaN 처리
        df_to_save = df_to_save.where(pd.notnull(df_to_save), None)

        print(f"저장할 데이터: {len(df_to_save)}건")

        # 중복 제거
        unique_cols = ['ticker', 'forecast_date', 'indicator', 'frequency', 'target_date']
        df_to_save = df_to_save.drop_duplicates(subset=unique_cols, keep='last')
        print(f"중복 제거 후: {len(df_to_save)}건")

        # valuation_time 설정 (고유값 보장)
        base_time = datetime.now()
        df_to_save = df_to_save.reset_index(drop=True)
        df_to_save['valuation_time'] = [base_time + timedelta(microseconds=i) for i in range(len(df_to_save))]

        # 테이블 스키마 확인 및 준비
        _ensure_korea_table_and_indexes(engine, table_name, db_info['database'])

        ticker = df_to_save['ticker'].iloc[0]
        forecast_date = df_to_save['forecast_date'].iloc[0]

        # 기존 데이터 삭제
        print("기존 데이터 삭제 중...")
        with engine.begin() as conn:
            delete_query = f"""DELETE FROM `{table_name}`
                             WHERE ticker = :ticker AND DATE(forecast_date) = DATE(:forecast_date)"""
            result = conn.execute(text(delete_query), {
                'ticker': ticker,
                'forecast_date': forecast_date
            })
            print(f"기존 데이터 {result.rowcount}건 삭제됨")

        # 새 데이터 삽입
        print("새 데이터 삽입 중...")
        with engine.begin() as conn:
            insert_query = text(f"""
                INSERT IGNORE INTO `{table_name}`
                (frequency, ticker, forecast_date, target_date, indicator, value, exog_var, params, valuation_time)
                VALUES (:frequency, :ticker, :forecast_date, :target_date, :indicator, :value, :exog_var, :params, :valuation_time)
            """)

            successful_inserts = 0
            for idx, row in df_to_save.iterrows():
                try:
                    result = conn.execute(insert_query, {
                        'frequency': row['frequency'],
                        'ticker': row['ticker'],
                        'forecast_date': row['forecast_date'],
                        'target_date': row['target_date'],
                        'indicator': row['indicator'],
                        'value': row['value'],
                        'exog_var': row['exog_var'],
                        'params': row['params'],
                        'valuation_time': row['valuation_time']
                    })
                    if result.rowcount > 0:
                        successful_inserts += 1
                except Exception as insert_error:
                    print(f"삽입 실패: {row['indicator']} - {insert_error}")

        # 결과 확인
        with engine.connect() as conn:
            today_count_query = text(f"""
                SELECT COUNT(*) FROM `{table_name}`
                WHERE ticker = :ticker AND DATE(forecast_date) = DATE(:forecast_date)
            """)
            today_count = conn.execute(today_count_query, {
                'ticker': ticker,
                'forecast_date': forecast_date
            }).scalar()

            print(f"저장 완료: {today_count}건 저장됨")

        engine.dispose()
        return True

    except Exception as e:
        print(f"저장 실패: {str(e)}")

        # 백업 저장
        try:
            backup_filename = f"korea_valuation_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
            combined_long_format.to_csv(backup_filename, index=False, encoding='utf-8-sig')
            print(f"백업 파일 저장됨: {backup_filename}")
        except Exception as backup_error:
            print(f"백업 저장 실패: {backup_error}")

        return False

# -----------------------------
# Save Results
# -----------------------------
print("\n8. 데이터베이스 저장...")
save_success = save_korea_valuation_to_db(db_info, combined_long_format)

if save_success:
    print("모든 작업이 성공적으로 완료되었습니다!")
else:
    print("예측은 완료되었으나 데이터베이스 저장에 실패했습니다. 메모리에는 데이터가 남아있습니다.")

# -----------------------------
# Final Summary
# -----------------------------
print("\n" + "=" * 50)
print("최종 요약:")
print(f"대상 기업: {tic_name}")
print(f"예측 기준일: {today_date.strftime('%Y-%m-%d')}")
print(f"외생변수: {hs_code if USE_EXOGENOUS else '사용 안함'}")
print(f"향후 4분기 매출 예측: {forward_revenue_without_exog/1e8:.1f}억원")
print(f"총 예측 레코드: {len(combined_long_format)}건")
print(f"예측 방법: SARIMA (분기별/월별)")
print(f"밸류에이션 로직: 미래 PSR × 미래 매출")
print(f"저장 테이블: korea_company_valuation_result")
print("=" * 50)

# 추가 유틸리티 함수들
def check_korea_database_status(db_info, ticker=tic_name):
    """한국 기업 밸류에이션 데이터베이스 상태 확인"""
    try:
        from sqlalchemy import create_engine, text

        connection_string = (
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        engine = create_engine(connection_string)
        table_name = 'korea_company_valuation_result'

        with engine.connect() as conn:
            # 전체 데이터 현황
            total_count = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`")).scalar()

            # 특정 ticker 데이터 현황
            ticker_count = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}` WHERE ticker = :ticker"),
                                      {'ticker': ticker}).scalar()

            # 최근 저장된 데이터
            recent_data = conn.execute(text(f"""
                SELECT forecast_date, COUNT(*) as count
                FROM `{table_name}`
                WHERE ticker = :ticker
                GROUP BY forecast_date
                ORDER BY forecast_date DESC
                LIMIT 5
            """), {'ticker': ticker}).fetchall()

            print(f"데이터베이스 현황:")
            print(f"  전체 레코드: {total_count:,}건")
            print(f"  {ticker} 레코드: {ticker_count:,}건")
            print(f"  최근 저장 이력:")
            for row in recent_data:
                print(f"    {row[0]}: {row[1]}건")

        engine.dispose()

    except Exception as e:
        print(f"상태 확인 실패: {e}")

def diagnose_data_quality():
    """데이터 품질 진단 함수"""
    print("\n=== 전체 데이터 품질 진단 ===")

    # 전역 변수들을 안전하게 체크
    global endog_df, quarterly_sum_df, endog_ratio_df, merged_revenue, merged_ratio_exog, combined_long_format

    # 주요 데이터프레임들 검증
    datasets = [
        ('매출 데이터', globals().get('endog_df')),
        ('외생변수 분기 집계', globals().get('quarterly_sum_df')),
        ('PSR 데이터', globals().get('endog_ratio_df')),
        ('매출 예측용 데이터', globals().get('merged_revenue')),
        ('PSR 예측용 데이터', globals().get('merged_ratio_exog')),
        ('최종 결과', globals().get('combined_long_format'))
    ]

    for name, df in datasets:
        if df is not None and hasattr(df, 'shape'):
            validate_dataframe(df, name)
        else:
            print(f"{name}: None (사용되지 않음 또는 미생성)")

print("\n=== 추가 유틸리티 함수 ===")
print("데이터 품질 진단: diagnose_data_quality()")
print("DB 상태 확인: check_korea_database_status(db_info, tic_name)")
print("\n한국 기업 밸류에이션 시스템 완료!")

sys.path에 등록된 경로: C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast
한국 기업 밸류에이션 시스템 시작
대상 기업: A000660
외생변수 HS코드: 854231
외생변수 사용: 예
1. 기본 데이터 로딩...
✅ 'korea_fs_data' 테이블에서 5902708건의 데이터를 가져왔습니다.

=== 매출 데이터 데이터 품질 체크 ===
매출 데이터 로딩 완료: 62건 (2010-03 ~ 2025-06)
2. 외생변수 데이터 준비...
✅ root_hs_code=854231에 해당하는 236개 행을 불러왔습니다.
외생변수 원본 데이터: 236건

=== 외생변수 원본 데이터 품질 체크 ===
final_expDlr_yoy: inf=0, NaN=12 (전체 236건 중)
expDlr_forecast_12m: inf=0, NaN=13 (전체 236건 중)

=== 분기별 집계 외생변수 데이터 품질 체크 ===
export_qoq_change: inf=0, NaN=1 (전체 79건 중)
export_yoy_change: inf=0, NaN=4 (전체 79건 중)

=== 스케일링된 외생변수 데이터 품질 체크 ===
export_qoq_change: inf=0, NaN=1 (전체 79건 중)
export_yoy_change: inf=0, NaN=4 (전체 79건 중)
외생변수 데이터 준비 완료: 79건
3. PSR 데이터 준비...


KeyboardInterrupt: 

In [ ]:
market_cap_df = fetch_table_data(db_info, "ks_listed_company_daily_marketcap")

In [4]:
# market_cap_df = fetch_table_data(db_info, "ks_listed_company_daily_marketcap")
market_cap_df = market_cap_df[market_cap_df['symbol'] == tic_name].copy()
market_cap_df['date'] = pd.to_datetime(market_cap_df['date'])

# 시가총액 데이터 정리
market_cap_df['market_cap'] = clean_numeric_data(market_cap_df['market_cap'], method='fill_median')

# PSR 계산을 위한 데이터 병합
merged_for_psr = pd.merge_asof(
    market_cap_df.sort_values('date'),
    endog_df.sort_values('date'),
    on='date',
    direction='backward'
)

# PSR 계산 (분모가 0이 되지 않도록 보호)
merged_for_psr['PSR'] = merged_for_psr['market_cap'] / (merged_for_psr[tic_name] + 1e-10)
merged_for_psr['PSR'] = clean_numeric_data(merged_for_psr['PSR'], method='fill_median')
merged_for_psr = merged_for_psr.dropna(subset=['PSR'])



KeyError: 'symbol'

In [6]:
test = market_cap_df[market_cap_df['ticker'] == tic_name].copy()

In [8]:
test[test['indicator'] == '시가총액']

,date,ticker,indicator,value
3831002,2020-01-01,A000660,시가총액,6.850500e+13
3833477,2020-01-02,A000660,시가총액,6.894180e+13
3835952,2020-01-03,A000660,시가총액,6.879620e+13
3838426,2020-01-06,A000660,시가총액,6.865060e+13
3840900,2020-01-07,A000660,시가총액,6.843220e+13
...,...,...,...,...
31244145,2025-08-11,A000660,시가총액,1.943770e+14
31247023,2025-08-12,A000660,시가총액,1.958330e+14
31249901,2025-08-13,A000660,시가총액,2.023850e+14
31252779,2025-08-14,A000660,시가총액,2.012930e+14
